In [1]:
import pandas as pd
import numpy as np

In [2]:
DATA_PATH = '../cleaned_data/combined_data_by_year.csv'

wage_columns = [
    'H_MEAN', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90',
    'A_MEAN', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90'
]

df = pd.read_csv(DATA_PATH, low_memory=False)
df[wage_columns] = df[wage_columns].apply(pd.to_numeric, errors='coerce')

for col in wage_columns:
    df[col] = df[col].fillna(df.groupby('STATE')[col].transform('mean'))
    df[col] = df[col].fillna(df.groupby('OCC_CODE')[col].transform('mean'))

In [3]:
df.head()

,AREA,ST,STATE,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,MEAN_PRSE,H_MEAN,...,H_PCT90,A_MEAN,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,year
0,1,AL,Alabama,11-0000,Management occupations,major,75100.0,1.0,0.6,44.060000,...,77.240000,91650.0,40540.0,56100.0,78870.0,112190.000000,160660.000000,NaN,NaN,2009
1,1,AL,Alabama,11-1011,Chief executives,NaN,2960.0,3.5,2.2,79.980000,...,32.448687,166350.0,70060.0,105360.0,160830.0,57433.100125,69393.365675,NaN,NaN,2009
2,1,AL,Alabama,11-1021,General and operations managers,NaN,33390.0,1.4,0.9,46.090000,...,32.448687,95880.0,42470.0,56280.0,79090.0,117360.000000,69393.365675,NaN,NaN,2009
3,1,AL,Alabama,11-1031,Legislators,NaN,1350.0,5.1,2.2,23.867328,...,32.448687,18840.0,14090.0,14720.0,15780.0,18590.000000,26680.000000,True,NaN,2009
4,1,AL,Alabama,11-2011,Advertising and promotions managers,NaN,160.0,14.5,7.3,39.450000,...,61.820000,82060.0,41170.0,55190.0,66250.0,90730.000000,128590.000000,NaN,NaN,2009


In [4]:
model_parameters = [
 'AREA',
 'ST',
 'OCC_CODE',
 'GROUP',
 'A_MEAN_prev_1y',
 'A_MEAN_prev_2y',
 'A_MEAN_prev_3y',
 'A_MEAN_prev_4y',
 'A_MEAN_prev_5y',
 'A_MEDIAN_prev_1y',
 'A_MEDIAN_prev_2y',
 'A_MEDIAN_prev_3y',
 'A_MEDIAN_prev_4y',
 'A_MEDIAN_prev_5y',
 'A_PCT10_prev_1y',
 'A_PCT10_prev_2y',
 'A_PCT10_prev_3y',
 'A_PCT10_prev_4y',
 'A_PCT10_prev_5y',
 'A_PCT25_prev_1y',
 'A_PCT25_prev_2y',
 'A_PCT25_prev_3y',
 'A_PCT25_prev_4y',
 'A_PCT25_prev_5y',
 'A_PCT75_prev_1y',
 'A_PCT75_prev_2y',
 'A_PCT75_prev_3y',
 'A_PCT75_prev_4y',
 'A_PCT75_prev_5y',
 'A_PCT90_prev_1y',
 'A_PCT90_prev_2y',
 'A_PCT90_prev_3y',
 'A_PCT90_prev_4y',
 'A_PCT90_prev_5y',
 'EMP_PRSE_prev_1y',
 'EMP_PRSE_prev_2y',
 'EMP_PRSE_prev_3y',
 'EMP_PRSE_prev_4y',
 'EMP_PRSE_prev_5y',
 'H_MEAN_prev_1y',
 'H_MEAN_prev_2y',
 'H_MEAN_prev_3y',
 'H_MEAN_prev_4y',
 'H_MEAN_prev_5y',
 'H_MEDIAN_prev_1y',
 'H_MEDIAN_prev_2y',
 'H_MEDIAN_prev_3y',
 'H_MEDIAN_prev_4y',
 'H_MEDIAN_prev_5y',
 'H_PCT10_prev_1y',
 'H_PCT10_prev_2y',
 'H_PCT10_prev_3y',
 'H_PCT10_prev_4y',
 'H_PCT10_prev_5y',
 'H_PCT25_prev_1y',
 'H_PCT25_prev_2y',
 'H_PCT25_prev_3y',
 'H_PCT25_prev_4y',
 'H_PCT25_prev_5y',
 'H_PCT75_prev_1y',
 'H_PCT75_prev_2y',
 'H_PCT75_prev_3y',
 'H_PCT75_prev_4y',
 'H_PCT75_prev_5y',
 'H_PCT90_prev_1y',
 'H_PCT90_prev_2y',
 'H_PCT90_prev_3y',
 'H_PCT90_prev_4y',
 'H_PCT90_prev_5y',
 'MEAN_PRSE_prev_1y',
 'MEAN_PRSE_prev_2y',
 'MEAN_PRSE_prev_3y',
 'MEAN_PRSE_prev_4y',
 'MEAN_PRSE_prev_5y',
 'TOT_EMP_prev_1y',
 'TOT_EMP_prev_2y',
 'TOT_EMP_prev_3y',
 'TOT_EMP_prev_4y',
 'TOT_EMP_prev_5y',
]

In [5]:
join_keys = ['STATE', 'OCC_CODE', 'year']
history_columns = [col for col in df.columns if col not in join_keys]

df_with_history = df.copy()

for lag in range(1, 6):
    history_df = df[join_keys + history_columns].copy()
    history_df['year'] = history_df['year'] + lag
    history_df = history_df.rename(
        columns={col: f'{col}_prev_{lag}y' for col in history_columns}
    )
    df_with_history = df_with_history.merge(
        history_df,
        on=join_keys,
        how='left'
    )

df_with_history = df_with_history.dropna(subset=model_parameters).reset_index(drop=True)

print(f'Original shape: {df.shape}')
print(f'With complete 5-year history shape: {df_with_history.shape}')
df_with_history.head()

Original shape: (739544, 24)
With complete 5-year history shape: (341705, 129)


,AREA,ST,STATE,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,MEAN_PRSE,H_MEAN,...,H_PCT75_prev_5y,H_PCT90_prev_5y,A_MEAN_prev_5y,A_PCT10_prev_5y,A_PCT25_prev_5y,A_MEDIAN_prev_5y,A_PCT75_prev_5y,A_PCT90_prev_5y,ANNUAL_prev_5y,HOURLY_prev_5y
0,1,AL,Alabama,11-0000,Management Occupations,major,96070.0,1.3,0.6,51.060000,...,63.180000,90.250000,110210.0,52470.0,70040.0,95870.0,131410.000000,187720.000000,NaN,NaN
1,1,AL,Alabama,11-1011,Chief Executives,detailed,690.0,11.4,4.8,72.240000,...,26.872431,32.448687,209300.0,98310.0,137140.0,196750.0,57433.100125,69393.365675,NaN,NaN
2,1,AL,Alabama,11-1021,General and Operations Managers,detailed,34370.0,2.7,0.9,54.500000,...,71.490000,32.448687,122420.0,55770.0,74270.0,103030.0,148710.000000,69393.365675,NaN,NaN
3,1,AL,Alabama,11-1031,Legislators,detailed,1030.0,10.0,4.5,23.867328,...,26.872431,32.448687,23390.0,16180.0,17070.0,18560.0,21590.000000,37320.000000,True,NaN
4,1,AL,Alabama,11-2021,Marketing Managers,detailed,1210.0,5.4,2.2,58.830000,...,88.800000,32.448687,143590.0,67650.0,85010.0,120270.0,184710.000000,69393.365675,NaN,NaN


In [6]:
df_with_history

,AREA,ST,STATE,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,MEAN_PRSE,H_MEAN,...,H_PCT75_prev_5y,H_PCT90_prev_5y,A_MEAN_prev_5y,A_PCT10_prev_5y,A_PCT25_prev_5y,A_MEDIAN_prev_5y,A_PCT75_prev_5y,A_PCT90_prev_5y,ANNUAL_prev_5y,HOURLY_prev_5y
0,1,AL,Alabama,11-0000,Management Occupations,major,96070.0,1.3,0.6,51.060000,...,63.180000,90.250000,110210.0,52470.0,70040.0,95870.0,131410.000000,187720.000000,NaN,NaN
1,1,AL,Alabama,11-1011,Chief Executives,detailed,690.0,11.4,4.8,72.240000,...,26.872431,32.448687,209300.0,98310.0,137140.0,196750.0,57433.100125,69393.365675,NaN,NaN
2,1,AL,Alabama,11-1021,General and Operations Managers,detailed,34370.0,2.7,0.9,54.500000,...,71.490000,32.448687,122420.0,55770.0,74270.0,103030.0,148710.000000,69393.365675,NaN,NaN
3,1,AL,Alabama,11-1031,Legislators,detailed,1030.0,10.0,4.5,23.867328,...,26.872431,32.448687,23390.0,16180.0,17070.0,18560.0,21590.000000,37320.000000,True,NaN
4,1,AL,Alabama,11-2021,Marketing Managers,detailed,1210.0,5.4,2.2,58.830000,...,88.800000,32.448687,143590.0,67650.0,85010.0,120270.0,184710.000000,69393.365675,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341700,78,VI,Virgin Islands,43-0000,Office and Administrative Support Occupations,major,8240.0,3.0,2.0,15.200000,...,13.470000,17.680000,23800.0,14760.0,17010.0,21410.0,28020.000000,36780.000000,NaN,NaN
341701,78,VI,Virgin Islands,47-0000,Construction and Extraction Occupations,major,2160.0,4.4,3.2,20.660000,...,19.030000,22.660000,32970.0,18760.0,22510.0,29670.0,39570.000000,47130.000000,NaN,NaN
341702,78,VI,Virgin Islands,49-0000,"Installation, Maintenance, and Repair Occupations",major,2390.0,5.8,3.0,20.870000,...,17.720000,22.100000,30030.0,17140.0,21010.0,27220.0,36860.000000,45960.000000,NaN,NaN
341703,78,VI,Virgin Islands,51-0000,Production Occupations,major,1640.0,3.7,7.9,20.910000,...,22.670000,26.790000,34140.0,15540.0,19180.0,31110.0,47150.000000,55730.000000,NaN,NaN


In [7]:
from sklearn.preprocessing import LabelEncoder

In [8]:
label_encoder = LabelEncoder()

for col in df_with_history.select_dtypes(include=['object']).columns:
    df_with_history[col] = label_encoder.fit_transform(df_with_history[col])

In [9]:
df_with_history

,AREA,ST,STATE,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,MEAN_PRSE,H_MEAN,...,H_PCT75_prev_5y,H_PCT90_prev_5y,A_MEAN_prev_5y,A_PCT10_prev_5y,A_PCT25_prev_5y,A_MEDIAN_prev_5y,A_PCT75_prev_5y,A_PCT90_prev_5y,ANNUAL_prev_5y,HOURLY_prev_5y
0,1,1,0,0,527,1,13968,13,6,51.060000,...,63.180000,90.250000,110210.0,52470.0,70040.0,95870.0,131410.000000,187720.000000,1,1
1,1,1,0,1,121,0,11863,34,245,72.240000,...,26.872431,32.448687,209300.0,98310.0,137140.0,196750.0,57433.100125,69393.365675,1,1
2,1,1,0,2,398,0,7558,127,9,54.500000,...,71.490000,32.448687,122420.0,55770.0,74270.0,103030.0,148710.000000,69393.365675,1,1
3,1,1,0,3,493,0,193,20,242,23.867328,...,26.872431,32.448687,23390.0,16180.0,17070.0,18560.0,21590.000000,37320.000000,0,1
4,1,1,0,5,533,0,1268,454,122,58.830000,...,88.800000,32.448687,143590.0,67650.0,85010.0,120270.0,184710.000000,69393.365675,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341700,78,48,48,533,622,1,12955,230,120,15.200000,...,13.470000,17.680000,23800.0,14760.0,17010.0,21410.0,28020.000000,36780.000000,1,1
341701,78,48,48,604,173,1,4787,344,229,20.660000,...,19.030000,22.660000,32970.0,18760.0,22510.0,29670.0,39570.000000,47130.000000,1,1
341702,78,48,48,667,462,1,5366,458,227,20.870000,...,17.720000,22.100000,30030.0,17140.0,21010.0,27220.0,36860.000000,45960.000000,1,1
341703,78,48,48,720,708,1,3138,237,276,20.910000,...,22.670000,26.790000,34140.0,15540.0,19180.0,31110.0,47150.000000,55730.000000,1,1


In [10]:
from sklearn.model_selection import train_test_split


In [11]:
X = df_with_history[model_parameters].values
y = df_with_history[model_parameters].values

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle = True
)

## Additional Models for Predicting Wage Levels

This section uses `A_MEAN` as the primary target because annual mean wages are slightly smoother and easier to interpret than hourly wages, while still being tightly related to `H_MEAN`. If you want to switch targets, change `target_col` below to `'H_MEAN'`.

The dataset is a strong fit for historical-feature modeling because it already contains five years of lagged wage and employment signals by occupation and state. To keep runtime manageable, the code below samples from the fully prepared history table when needed.


In [16]:
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, silhouette_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

target_col = 'A_MEAN'
feature_cols = model_parameters + ['year']

model_df = df_with_history[feature_cols + [target_col]].dropna().copy()

# Re-encode only if object columns remain. Earlier cells already encode the notebook dataframe.
for col in model_df.select_dtypes(include=['object']).columns:
    model_df[col] = LabelEncoder().fit_transform(model_df[col].astype(str))

sample_size = min(120000, len(model_df))
model_sample = model_df.sample(sample_size, random_state=42).reset_index(drop=True)

X = model_sample[feature_cols]
y = model_sample[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = {}
print(f'Target: {target_col}')
print(f'Modeling rows used: {len(model_sample):,}')
print(f'Feature count: {len(feature_cols)}')


Target: A_MEAN
Modeling rows used: 120,000
Feature count: 80


### 1. Regression Model: Random Forest Regressor

**Why this model was chosen**
- The wage target is continuous, so regression is the most direct predictive setup.
- Random forests work well with many lagged numeric predictors and non-linear relationships.
- They are also robust when a few encoded categorical columns are mixed with many historical wage columns.

**Model assumptions**
- Random forests do not assume linearity.
- They assume the training data is representative of the future patterns we want to predict.
- They can overfit if trees become too deep, so depth and leaf size still matter.

**Hyperparameter tuning**
- `n_estimators`: more trees usually improve stability.
- `max_depth`: limits overfitting on very detailed historical patterns.
- `min_samples_leaf`: smooths predictions by preventing tiny terminal nodes.

**Challenges faced and solution**
- The dataset is large, so exhaustive search would be slow. I used a compact manual search grid with sampled rows for practical tuning.


In [31]:
rf_candidates = [
    {'n_estimators': 80, 'max_depth': 16, 'min_samples_leaf': 1},
    {'n_estimators': 120, 'max_depth': 20, 'min_samples_leaf': 1},
    {'n_estimators': 120, 'max_depth': None, 'min_samples_leaf': 3},
]

rf_runs = []
for params in rf_candidates:
    rf_model = RandomForestRegressor(random_state=42, n_jobs=1, **params)
    rf_model.fit(X_train, y_train)
    preds = rf_model.predict(X_test)
    rf_runs.append({
        **params,
        'rmse': np.sqrt(mean_squared_error(y_test, preds)),
        'mse': mean_squared_error(y_test, preds),
        # 'rmse': mean_squared_error(y_test, preds, squared=False),
        'r2': r2_score(y_test, preds)
    })

rf_results = pd.DataFrame(rf_runs).sort_values(['r2', 'rmse', 'mse'], ascending=[False, True, True]).reset_index(drop=True)
best_rf_params = rf_results.loc[0, ['n_estimators', 'max_depth', 'min_samples_leaf']].to_dict()
results['Random Forest Regressor'] = {
    'task': 'Regression',
    'best_r2': rf_results.loc[0, 'r2'],
    'best_rmse': rf_results.loc[0, 'rmse'],
    'best_mse': rf_results.loc[0, 'mse']
}

display(rf_results)


,n_estimators,max_depth,min_samples_leaf,rmse,mse,r2
0,120,NaN,3,5233.671109,2.739131e+07,0.970788
1,80,16.0,1,5300.745741,2.809791e+07,0.970035
2,120,20.0,1,5354.859413,2.867452e+07,0.969420


In [32]:
MODEL_PATH = '../models/rf_model.cpickle'

import _pickle as cPickle
# Saving the model
with open(MODEL_PATH, 'wb') as f:
    cPickle.dump(rf_model, f)

# To load and read the model

with open(MODEL_PATH, 'rb') as f:
    rf_model = cPickle.load(f)

# preds = rf_model.predict(X_test)

### 2. Classification Model: Decision Tree Classifier

**Why this model was chosen**
- Classification is useful when the business question is wage tier prediction instead of exact wage prediction.
- A decision tree is easy to explain because it produces explicit splits on lagged wage and employment features.
- Trees also handle non-linear thresholds naturally, which fits wage bands well.

**Model assumptions**
- Decision trees assume the target classes can be separated by recursive feature splits.
- They do not assume normality or linearity.
- Deep trees can memorize noise, so pruning-related settings are important.

**Hyperparameter tuning**
- `max_depth`: controls tree complexity.
- `min_samples_leaf`: reduces unstable tiny leaves.
- `criterion`: compares `gini` vs `entropy`.

**Challenges faced and solution**
- `A_MEAN` is continuous, so I converted it into quartile-based wage bands to make classification meaningful while keeping the classes balanced.


In [35]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

y_band = pd.qcut(y, q=4, labels=['low', 'mid_low', 'mid_high', 'high'], duplicates='drop')

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_band, test_size=0.2, random_state=42, stratify=y_band
)

tree_candidates = [
    {'max_depth': 6,    'min_samples_leaf': 25, 'criterion': 'gini'},
    {'max_depth': 10,   'min_samples_leaf': 25, 'criterion': 'entropy'},
    {'max_depth': None, 'min_samples_leaf': 50, 'criterion': 'entropy'},
]

classes = ['low', 'mid_low', 'mid_high', 'high']

tree_runs = []
for params in tree_candidates:
    tree_model = DecisionTreeClassifier(random_state=42, **params)
    tree_model.fit(Xc_train, yc_train)
    preds      = tree_model.predict(Xc_test)
    proba      = tree_model.predict_proba(Xc_test)          # shape (n, 4)

    # Binarize true labels for one-vs-rest ROC-AUC
    yc_test_bin = label_binarize(yc_test, classes=classes)  # shape (n, 4)

    tree_runs.append({
        **params,
        'accuracy':  accuracy_score(yc_test, preds),
        'precision': precision_score(yc_test, preds, average='weighted', zero_division=0),
        'recall':    recall_score(yc_test, preds,    average='weighted', zero_division=0),
        'f1':        f1_score(yc_test, preds,        average='weighted', zero_division=0),
        'roc_auc':   roc_auc_score(yc_test_bin, proba, average='weighted', multi_class='ovr'),
    })

tree_results = (
    pd.DataFrame(tree_runs)
      .sort_values('accuracy', ascending=False)
      .reset_index(drop=True)
)

results['Decision Tree Classifier'] = {
    'task':          'Classification',
    'best_accuracy': tree_results.loc[0, 'accuracy'],
    'best_f1':       tree_results.loc[0, 'f1'],        # optional: track best F1 too
}

display(tree_results)

,max_depth,min_samples_leaf,criterion,accuracy,precision,recall,f1,roc_auc
0,6.0,25,gini,0.899833,0.900190,0.899833,0.899947,0.466786
1,NaN,50,entropy,0.896750,0.896613,0.896750,0.896677,0.482633
2,10.0,25,entropy,0.894083,0.894409,0.894083,0.894092,0.487806


In [36]:
MODEL_PATH = '../models/tree_model.cpickle'

import _pickle as cPickle
# Saving the model
with open(MODEL_PATH, 'wb') as f:
    cPickle.dump(tree_model, f)

# To load and read the model

with open(MODEL_PATH, 'rb') as f:
    tree_model = cPickle.load(f)

# preds = tree_model.predict(X_test)

### 3. Clustering Model: K-Means

**Why this model was chosen**
- Clustering is useful for discovering wage segments before supervised modeling.
- K-Means is a strong baseline because the dataset has many numeric lag features that can be standardized and grouped into broad labor-market profiles.
- The cluster labels can later become engineered features for a predictive model.

**Model assumptions**
- K-Means assumes roughly spherical clusters in scaled feature space.
- It is sensitive to feature scale, so standardization is required.
- It requires choosing the number of clusters in advance.

**Hyperparameter tuning**
- `n_clusters`: tested a small range of cluster counts.
- `n_init`: increased to make initialization more stable.
- Silhouette score is used to compare cluster quality.

**Challenges faced and solution**
- Unsupervised models do not directly optimize `A_MEAN`, so I summarize each cluster by its average target value to make the result interpretable for prediction work.


In [37]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

cluster_features = [
    'A_MEAN_prev_1y', 'A_MEAN_prev_2y', 'H_MEAN_prev_1y',
    'TOT_EMP_prev_1y', 'EMP_PRSE_prev_1y', 'MEAN_PRSE_prev_1y'
]

cluster_df = model_sample[cluster_features + [target_col]].dropna().sample(
    min(25000, len(model_sample)), random_state=42
).copy()

scaler = StandardScaler()
cluster_X = scaler.fit_transform(cluster_df[cluster_features])

kmeans_runs = []
for k in [3, 4, 5, 6]:
    km_model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km_model.fit_predict(cluster_X)
    run_df = cluster_df.copy()
    run_df['cluster'] = labels
    kmeans_runs.append({
        'k': k,
        'silhouette':      silhouette_score(cluster_X, labels),
        'davies_bouldin':  davies_bouldin_score(cluster_X, labels),   # ← added
        'avg_target_std':  run_df.groupby('cluster')[target_col].mean().std()
    })

# Lower davies_bouldin is better, so sort it ascending
kmeans_results = (
    pd.DataFrame(kmeans_runs)
      .sort_values(['silhouette', 'davies_bouldin', 'avg_target_std'],
                   ascending=[False, True, False])
      .reset_index(drop=True)
)

best_k = int(kmeans_results.loc[0, 'k'])
best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_df['cluster'] = best_kmeans.fit_predict(cluster_X)
cluster_profile = cluster_df.groupby('cluster')[cluster_features + [target_col]].mean().round(2)

results['K-Means'] = {
    'task':             'Clustering',
    'best_k':           best_k,
    'best_silhouette':  kmeans_results.loc[0, 'silhouette'],
    'best_db_index':    kmeans_results.loc[0, 'davies_bouldin'],   # ← added
}

display(kmeans_results)
display(cluster_profile)

,k,silhouette,davies_bouldin,avg_target_std
0,3,0.235209,1.480709,36124.761101
1,4,0.235092,1.356977,57926.453201
2,5,0.234067,1.298854,55222.714696
3,6,0.227991,1.209893,52234.422431


,A_MEAN_prev_1y,A_MEAN_prev_2y,H_MEAN_prev_1y,TOT_EMP_prev_1y,EMP_PRSE_prev_1y,MEAN_PRSE_prev_1y,A_MEAN
cluster,,,,,,,
0,44709.86,43630.87,21.19,8384.20,109.75,178.84,46036.03
1,45633.42,44527.35,21.62,8529.71,465.18,106.68,46916.05
2,107141.49,104431.52,49.51,8498.91,275.77,146.13,109041.32


### 4. Frequent Pattern Mining Model: FP-Growth

**Why this model was chosen**
- Frequent pattern mining is useful here because wage outcomes may be tied to recurring combinations of occupation group, geography, and lagged pay bands.
- FP-Growth is more scalable than Apriori on larger transaction-style datasets.
- It helps reveal interpretable patterns associated with high-pay outcomes even though it is not a direct regressor.

**Model assumptions**
- The data must be converted into basket-style boolean items.
- The method assumes useful structure exists in co-occurring categories and discretized numeric ranges.
- Support and confidence thresholds strongly affect which patterns appear.

**Hyperparameter tuning**
- `min_support`: controls how common a pattern must be.
- `confidence` threshold in rule generation controls how reliable a rule must be.
- Numeric features are binned into quartiles so they can be mined as items.

**Challenges faced and solution**
- FP-Growth cannot use raw continuous values, so I discretized the important lagged wage and employment fields into interpretable bands.


In [28]:
pattern_df = model_sample[[
    'ST', 'GROUP', target_col, 'TOT_EMP_prev_1y', 'H_MEAN_prev_1y'
]].dropna().copy()

pattern_df['target_band'] = pd.qcut(
    pattern_df[target_col], q=4, labels=['low_pay', 'mid_low_pay', 'mid_high_pay', 'high_pay'], duplicates='drop'
)
pattern_df['emp_band'] = pd.qcut(
    pattern_df['TOT_EMP_prev_1y'], q=4, labels=['small_emp', 'mid_small_emp', 'mid_large_emp', 'large_emp'], duplicates='drop'
)
pattern_df['hist_hourly_band'] = pd.qcut(
    pattern_df['H_MEAN_prev_1y'], q=4, labels=['low_hist_hourly', 'mid_low_hist_hourly', 'mid_high_hist_hourly', 'high_hist_hourly'], duplicates='drop'
)

top_states = pattern_df['ST'].value_counts().head(8).index
top_groups = pattern_df['GROUP'].value_counts().head(5).index

basket = pd.DataFrame({
    'high_pay': pattern_df['target_band'].eq('high_pay'),
    'large_emp': pattern_df['emp_band'].eq('large_emp'),
    'high_hist_hourly': pattern_df['hist_hourly_band'].eq('high_hist_hourly')
})

for state in top_states:
    basket[f'STATE_{state}'] = pattern_df['ST'].eq(state)

for group in top_groups:
    basket[f'GROUP_{group}'] = pattern_df['GROUP'].eq(group)

itemsets = fpgrowth(basket.astype(bool), min_support=0.05, use_colnames=True)
rules = association_rules(itemsets, metric='confidence', min_threshold=0.60)
high_pay_rules = rules[rules['consequents'].apply(lambda s: 'high_pay' in s)].copy()
high_pay_rules = high_pay_rules.sort_values(['lift', 'confidence'], ascending=False)

results['FP-Growth'] = {
    'task': 'Frequent Pattern Mining',
    'rule_count_to_high_pay': len(high_pay_rules),
    'best_lift': high_pay_rules['lift'].iloc[0] if len(high_pay_rules) else np.nan
}

display(high_pay_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


,antecedents,consequents,support,confidence,lift
2,(high_hist_hourly),(high_pay),0.211242,0.845220,3.380881
4,"(GROUP_0, high_hist_hourly)",(high_pay),0.199208,0.843746,3.374982
7,(high_hist_hourly),"(GROUP_0, high_pay)",0.199208,0.797072,3.350100
12,"(high_hist_hourly, large_emp)",(high_pay),0.052150,0.835068,3.340272


### Model Comparison

The regression model should usually be the best choice for exact wage prediction because it directly optimizes a continuous target. The classification, clustering, and FP-growth models are still useful because they add interpretable wage tiers, segments, and recurring high-pay patterns that help explain the target from different angles.


In [30]:
pd.DataFrame(results).T


,task,best_r2,best_rmse,best_accuracy,best_k,best_silhouette,rule_count_to_high_pay,best_lift
Random Forest Regressor,Regression,0.970788,5233.671109,NaN,NaN,NaN,NaN,NaN
Decision Tree Classifier,Classification,NaN,NaN,0.899833,NaN,NaN,NaN,NaN
K-Means,Clustering,NaN,NaN,NaN,3,0.235209,NaN,NaN
FP-Growth,Frequent Pattern Mining,NaN,NaN,NaN,NaN,NaN,4,3.380881
